In [1]:
#H0: There is no linear relationship between GDP per capita and depression rate.
#H1: There is a linear relationship.
from scipy.stats import pearsonr
import pandas as pd

df = pd.read_csv("../data/clean/mental_econ_merged.csv")

r, p = pearsonr(df['gdp_per_capita'], df['depression_rate'])

print("Pearson correlation:", r)
print("p-value:", p)


Pearson correlation: 0.13908226617331176
p-value: 4.0585415762294175e-21


In [4]:
#H0: GDP has no effect on depression rate (β = 0).
#H1: GDP affects depression (β ≠ 0).
import statsmodels.formula.api as smf
model = smf.ols("depression_rate ~ gdp_per_capita", data=df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        depression_rate   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     89.83
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           4.06e-21
Time:                        18:26:16   Log-Likelihood:                -4510.6
No. Observations:                4556   AIC:                             9025.
Df Residuals:                    4554   BIC:                             9038.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          3.4355      0.013    268.571      0.000       3.410       3.461
gdp_per_capita  5.851e-06   6.17e-07      9.478      0.000    4.64e-06    7.06e-06
==============================================================================
Omnibus:                      158.361   Durbin-Watson:                   0.069
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              180.596
Skew:                           0.436   Prob(JB):                     6.08e-40
Kurtosis:                       3.435   Cond. No.                     2.75e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.75e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [7]:
#Because GDP is extremely right-skewed, we test:
import numpy as np
df['log_gdp'] = np.log(df['gdp_per_capita'] + 1)

model2 = smf.ols("depression_rate ~ log_gdp", data=df).fit()
model2.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        depression_rate   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     4.341
Date:                Sun, 30 Nov 2025   Prob (F-statistic):             0.0373
Time:                        18:28:02   Log-Likelihood:                -4552.9
No. Observations:                4556   AIC:                             9110.
Df Residuals:                    4554   BIC:                             9123.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.3677      0.071     47.175      0.000       3.228       3.508
log_gdp        0.0166      0.008      2.084      0.037       0.001       0.032
==============================================================================
Omnibus:                      130.897   Durbin-Watson:                   0.068
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              144.520
Skew:                           0.402   Prob(JB):                     4.15e-32
Kurtosis:                       3.340   Cond. No.                         66.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [8]:
df['gdp_group'] = pd.qcut(df['gdp_per_capita'], 2, labels=['Low GDP', 'High GDP'])



In [9]:
from scipy.stats import ttest_ind

low = df[df['gdp_group'] == 'Low GDP']['depression_rate']
high = df[df['gdp_group'] == 'High GDP']['depression_rate']

t, p = ttest_ind(low, high)
print("t-test:", t, "p-value:", p)


t-test: -2.2455633676295967 p-value: 0.0247797610905548


In [10]:
model3 = smf.ols("depression_rate ~ year", data=df).fit()
model3.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        depression_rate   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     7.171
Date:                Sun, 30 Nov 2025   Prob (F-statistic):            0.00744
Time:                        18:29:07   Log-Likelihood:                -4551.5
No. Observations:                4556   AIC:                             9107.
Df Residuals:                    4554   BIC:                             9120.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.9717      2.411      4.136      0.000       5.245      14.699
year          -0.0032      0.001     -2.678      0.007      -0.006      -0.001
==============================================================================
Omnibus:                      133.605   Durbin-Watson:                   0.067
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              147.542
Skew:                           0.407   Prob(JB):                     9.15e-33
Kurtosis:                       3.337   Cond. No.                     4.96e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.96e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [13]:
df = pd.read_csv("../data/clean/mental_econ_inequality_full.csv")
print(df.columns.tolist())


['country', 'Code', 'year', 'depression_rate', 'gdp_per_capita', 'gini_index', 'population', 'gini_index_owid', 'avg_income_usd', 'top10_share', 'bottom10_share', 'income_group']


In [15]:
from scipy.stats import pearsonr

tests = {
    "Gini (dataset 1)": ("gini_index", "depression_rate"),
    "Gini (OWID)": ("gini_index_owid", "depression_rate"),
    "Top 10% Share": ("top10_share", "depression_rate"),
    "Bottom 10% Share": ("bottom10_share", "depression_rate"),
    "Average Income (USD)": ("avg_income_usd", "depression_rate"),
    "GDP per capita": ("gdp_per_capita", "depression_rate")
}

for name, (x, y) in tests.items():
    # drop rows where either variable is missing
    clean_df = df[[x, y]].dropna()
    r, p = pearsonr(clean_df[x], clean_df[y])
    print(f"{name:20s}  r = {r: .4f},  p = {p: .4e}")


Gini (dataset 1)      r = -0.2708,  p =  8.6319e-25
Gini (OWID)           r = -0.0069,  p =  9.0962e-01
Top 10% Share         r = -0.0865,  p =  1.5651e-01
Bottom 10% Share      r = -0.1128,  p =  6.4127e-02
Average Income (USD)  r =  0.0002,  p =  9.9695e-01
GDP per capita        r =  0.1391,  p =  4.0585e-21


In [ ]:
#H0: Depression rates are the same in low vs high inequality countries.
#H1: Depression rates differ.

from scipy.stats import ttest_ind
import pandas as pd

# create two groups by median inequality
df_ineq = df.dropna(subset=["gini_index_owid", "depression_rate"]).copy()

df_ineq["ineq_group"] = pd.qcut(
    df_ineq["gini_index_owid"],
    q=2,
    labels=["Low inequality", "High inequality"]
)

low = df_ineq[df_ineq["ineq_group"]=="Low inequality"]["depression_rate"]
high = df_ineq[df_ineq["ineq_group"]=="High inequality"]["depression_rate"]

t, p = ttest_ind(low, high)
print("t-statistic:", t)
print("p-value   :", p)
print("Mean depression (low): ", low.mean())
print("Mean depression (high):", high.mean())


t-statistic: 0.06916503314276408
p-value   : 0.9449098335816176
Mean depression (low):  3.8511932666666664
Mean depression (high): 3.8465834962962973
